# LDP Tax Pipeline — Query Notebook
**Spark 4.1.1 | Medallion Architecture: Bronze → Silver → Gold**

Reads the four materialized views written by the SDP pipeline (`./run.sh`) from the local warehouse.

| Layer | Table | Description |
|-------|-------|-------------|
| Bronze | `bronze_tax_records` | Raw ingestion from S3, schema-on-read |
| Silver | `silver_tax_records` | Cleansed, typed, deduped, + effective_tax_rate |
| Gold 1 | `gold_tax_annual_summary` | Aggregated by (tax_year, state, filing_status) |
| Gold 2 | `gold_tax_income_bands` | Distribution across income & effective-rate bands |

---
## Setup — SparkSession & Table Registration

In [ ]:
import os, sys, subprocess

# ── Mode toggle ────────────────────────────────────────────────────────────────
# "cloud" → reads from Azure ADLS Gen2 via adlfs + pyarrow (no JVM needed)
# "local" → reads from ./spark-warehouse  (after LOCAL_MODE=true ./run.sh)
MODE = "cloud"

SCRIPT_DIR = "/Volumes/D/Projects/SparkSDP"

# ── Load .env ─────────────────────────────────────────────────────────────────
def _load_dotenv(path):
    env = {}
    if not os.path.exists(path):
        return env
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env          = _load_dotenv(os.path.join(SCRIPT_DIR, ".env"))
ADLS_ACCOUNT = env.get("ADLS_ACCOUNT_NAME") or os.environ.get("ADLS_ACCOUNT_NAME", "")
ADLS_KEY     = env.get("ADLS_ACCOUNT_KEY")  or os.environ.get("ADLS_ACCOUNT_KEY",  "")

# ── Install deps if needed ─────────────────────────────────────────────────────
for pkg in ("adlfs", "duckdb"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import adlfs, duckdb
import pyarrow.parquet as pq
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)

# ── Load tables from ADLS or local ────────────────────────────────────────────
TABLES = ["bronze_tax_records", "silver_tax_records",
          "gold_tax_annual_summary", "gold_tax_income_bands"]

con = duckdb.connect()
dfs = {}
missing = []

if MODE == "cloud":
    if not ADLS_ACCOUNT or not ADLS_KEY:
        raise ValueError("ADLS_ACCOUNT_NAME and ADLS_ACCOUNT_KEY must be set in .env")
    fs        = adlfs.AzureBlobFileSystem(account_name=ADLS_ACCOUNT, account_key=ADLS_KEY)
    CONTAINER = "medallion"
    PREFIX    = "ldp-tax/warehouse"
    for t in TABLES:
        try:
            df = pq.read_table(f"{CONTAINER}/{PREFIX}/{t}", filesystem=fs).to_pandas()
            con.register(t, df)
            dfs[t] = df
            print(f"[OK] {t}  ({len(df)} rows)")
        except Exception as e:
            missing.append(t)
            print(f"[WARN] {t}: {e}")
    warehouse_display = f"abfs://{CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net/{PREFIX}"
else:
    for t in TABLES:
        path = os.path.join(SCRIPT_DIR, "spark-warehouse", t)
        try:
            df = pq.read_table(path).to_pandas()
            con.register(t, df)
            dfs[t] = df
            print(f"[OK] {t}  ({len(df)} rows)")
        except Exception as e:
            missing.append(t)
            print(f"[WARN] {t}: {e}")
    warehouse_display = os.path.join(SCRIPT_DIR, "spark-warehouse")

# ── Named DataFrame references ─────────────────────────────────────────────────
bronze       = dfs.get("bronze_tax_records",      pd.DataFrame())
silver       = dfs.get("silver_tax_records",      pd.DataFrame())
gold_summary = dfs.get("gold_tax_annual_summary", pd.DataFrame())
gold_bands   = dfs.get("gold_tax_income_bands",   pd.DataFrame())

# ── spark-compatible wrapper for SQL cells (no JVM) ───────────────────────────
class _Frame:
    def __init__(self, df): self._df = df
    def show(self, n=20, truncate=True):
        df = self._df.head(n)
        total = len(self._df)
        with pd.option_context("display.max_colwidth", None if not truncate else 20,
                               "display.width", 200):
            print(df.to_string(index=False))
        if total > n:
            print(f"only showing top {n} rows\n")
        else:
            print()

class _Session:
    version = f"DuckDB {duckdb.__version__} (no JVM)"
    def sql(self, query): return _Frame(con.execute(query).df())

spark = _Session()

print(f"\n{spark.version} | mode={MODE}")
print(f"Warehouse : {warehouse_display}")
if missing:
    print(f"Missing   : {', '.join(missing)}  ← run ./run.sh first")

---
## Bronze Layer — Raw Ingestion

In [ ]:
# All raw records — exactly as read from S3
spark.sql("""
    SELECT taxpayer_id, taxpayer_name, tax_year, filing_status, state,
           gross_income, tax_liability, filing_date, return_type,
           _ingested_at
    FROM   bronze_tax_records
    ORDER BY tax_year DESC, taxpayer_id
""").show(25, truncate=False)

In [ ]:
# Row count and date range
spark.sql("""
    SELECT COUNT(*)              AS total_rows,
           MIN(tax_year)         AS earliest_year,
           MAX(tax_year)         AS latest_year,
           COUNT(DISTINCT state) AS distinct_states,
           MIN(_ingested_at)     AS first_ingestion,
           MAX(_ingested_at)     AS last_ingestion
    FROM   bronze_tax_records
""").show(truncate=False)

---
## Silver Layer — Cleansed & Validated

In [ ]:
# All cleansed records with effective tax rate
cols = ["taxpayer_id", "taxpayer_name", "tax_year", "filing_status", "state",
        "gross_income", "taxable_income", "deductions",
        "tax_liability", "effective_tax_rate",
        "balance_due", "refund_amount", "filing_date"]

(silver[cols]
 .sort_values(["tax_year", "effective_tax_rate"], ascending=[False, False])
 .head(25))

In [ ]:
# Data quality summary per year
(silver
 .groupby("tax_year")
 .agg(
     total_records       = ("taxpayer_id",       "count"),
     owe_tax             = ("balance_due",        lambda x: (x > 0).sum()),
     get_refund          = ("refund_amount",      lambda x: (x > 0).sum()),
     avg_eff_rate_pct    = ("effective_tax_rate", lambda x: round(x.mean(), 2)),
     total_tax_collected = ("tax_liability",      lambda x: round(x.sum(), 2)),
     total_refunds       = ("refund_amount",      lambda x: round(x.sum(), 2)),
 )
 .sort_values("tax_year", ascending=False)
 .reset_index())

In [ ]:
# Top 10 highest effective tax rates
(silver[["taxpayer_id", "taxpayer_name", "tax_year", "state",
         "gross_income", "tax_liability", "effective_tax_rate"]]
 .sort_values("effective_tax_rate", ascending=False)
 .head(10))

In [ ]:
# Taxpayers who owe more tax (balance_due > 0)
(silver
 .loc[silver["balance_due"] > 0,
      ["taxpayer_id", "taxpayer_name", "tax_year", "state",
       "gross_income", "tax_liability", "tax_paid", "balance_due"]]
 .sort_values("balance_due", ascending=False))

In [ ]:
# Taxpayers getting a refund (refund_amount > 0)
(silver
 .loc[silver["refund_amount"] > 0,
      ["taxpayer_id", "taxpayer_name", "tax_year", "state",
       "gross_income", "tax_paid", "refund_amount"]]
 .sort_values("refund_amount", ascending=False))

In [ ]:
# YoY comparison for taxpayers who filed multiple years
multi_year_ids = silver.groupby("taxpayer_id").filter(lambda x: len(x) > 1)["taxpayer_id"].unique()

(silver
 .loc[silver["taxpayer_id"].isin(multi_year_ids),
      ["taxpayer_id", "taxpayer_name", "tax_year",
       "gross_income", "tax_liability", "effective_tax_rate"]]
 .sort_values(["taxpayer_id", "tax_year"]))

---
## Gold Layer 1 — Annual Tax Summary

In [ ]:
# Full annual summary — all years, all states
cols = ["tax_year", "state", "filing_status", "taxpayer_count",
        "total_gross_income", "total_tax_liability",
        "avg_effective_tax_rate", "total_balance_due",
        "total_refund_amount", "pct_with_balance_due"]

(gold_summary[cols]
 .sort_values(["tax_year", "total_tax_liability"], ascending=[False, False])
 .head(30))

In [ ]:
# Top states by total tax collected (all years combined)
(gold_summary
 .groupby("state")
 .agg(
     total_taxpayers     = ("taxpayer_count",        "sum"),
     total_tax_collected = ("total_tax_liability",   lambda x: round(x.sum(), 2)),
     avg_eff_rate_pct    = ("avg_effective_tax_rate", lambda x: round(x.mean(), 2)),
     total_refunds       = ("total_refund_amount",   lambda x: round(x.sum(), 2)),
 )
 .sort_values("total_tax_collected", ascending=False)
 .reset_index())

In [ ]:
# Filing status breakdown — avg income and effective rate
(gold_summary
 .groupby("filing_status")
 .agg(
     taxpayers         = ("taxpayer_count",        "sum"),
     avg_gross_income  = ("avg_gross_income",       lambda x: round(x.mean(), 2)),
     avg_eff_rate_pct  = ("avg_effective_tax_rate", lambda x: round(x.mean(), 2)),
     total_refunds     = ("total_refund_amount",    lambda x: round(x.sum(), 2)),
     total_balance_due = ("total_balance_due",      lambda x: round(x.sum(), 2)),
 )
 .sort_values("avg_eff_rate_pct", ascending=False)
 .reset_index())

In [ ]:
# States where more than 50% of taxpayers have a balance due
(gold_summary
 .loc[gold_summary["pct_with_balance_due"] > 50,
      ["state", "filing_status", "taxpayer_count",
       "taxpayers_with_balance_due", "pct_with_balance_due"]]
 .sort_values("pct_with_balance_due", ascending=False))

---
## Gold Layer 2 — Income Band Analysis

In [ ]:
# Full income band breakdown for 2024
cols = ["income_band", "effective_rate_band", "taxpayer_count",
        "avg_gross_income", "avg_taxable_income", "avg_deductions",
        "avg_tax_liability", "avg_effective_tax_rate"]

(gold_bands
 .loc[gold_bands["tax_year"] == 2024, cols]
 .sort_values(["income_band", "effective_rate_band"])
 .head(20))

In [ ]:
# Total tax liability rolled up by income band (all years)
(gold_bands
 .groupby("income_band")
 .agg(
     total_taxpayers     = ("taxpayer_count",        "sum"),
     total_tax_liability = ("total_tax_liability",   lambda x: round(x.sum(), 2)),
     avg_eff_rate_pct    = ("avg_effective_tax_rate", lambda x: round(x.mean(), 2)),
     total_refunds       = ("total_refund_amount",   lambda x: round(x.sum(), 2)),
 )
 .sort_values("income_band")
 .reset_index())

In [ ]:
# What share of total tax does each income band contribute?
grand_total = gold_bands["total_tax_liability"].sum()

(gold_bands
 .groupby("income_band")
 .agg(
     taxpayers     = ("taxpayer_count",      "sum"),
     tax_liability = ("total_tax_liability", lambda x: round(x.sum(), 2)),
 )
 .assign(pct_of_total_tax=lambda df: (df["tax_liability"] / grand_total * 100).round(2))
 .sort_values("income_band")
 .reset_index())

---
## Ad-hoc Queries — Edit & Run

In [ ]:
# Edit this cell for any custom query
spark.sql("""
    SELECT *
    FROM   silver_tax_records
    WHERE  state = 'CA'
    ORDER BY tax_year DESC, gross_income DESC
""").show(truncate=False)

In [ ]:
# Another scratch cell
spark.sql("""
    SELECT *
    FROM   gold_tax_annual_summary
    WHERE  tax_year = 2024
      AND  state = 'TX'
""").show(truncate=False)

---
## Cleanup

In [ ]:
con.close()
print("DuckDB connection closed.")